In [5]:

import sys
from pathlib import Path

# Add project root to path
sys.path.insert(0, str(Path.cwd().parent))

from ADoptEX.benchmark.methods import get_standard_methods
from ADoptEX.benchmark.runner import run_benchmark
from ADoptEX.benchmark.scenarios import get_synthetic_scenarios

DEFAULT_RESULTS_DIR = Path.cwd() / "results_interactive"


In [6]:
# Configuration
resume = False
n_starts = 10

scenario = "tonic_15pct_membrane"
method = "grad_guarino"

# Build method list 
scenarios = list(get_synthetic_scenarios(n_starts=n_starts))

print("Available scenarios:")
for s in get_synthetic_scenarios():
    print(f"  {s.name}")


Available scenarios:
  tonic_15pct_membrane
  tonic_25pct_membrane
  tonic_15pct_full
  tonic_25pct_full
  adaptation_15pct_membrane
  adaptation_25pct_membrane
  adaptation_15pct_full
  adaptation_25pct_full
  initial_bursting_15pct_membrane
  initial_bursting_25pct_membrane
  initial_bursting_15pct_full
  initial_bursting_25pct_full
  regular_bursting_15pct_membrane
  regular_bursting_25pct_membrane
  regular_bursting_15pct_full
  regular_bursting_25pct_full
  delayed_accelerating_15pct_membrane
  delayed_accelerating_25pct_membrane
  delayed_accelerating_15pct_full
  delayed_accelerating_25pct_full


In [7]:
from ADoptEX.training import TrainingConfig
from ADoptEX.loss import VanRossumLossConfig
from ADoptEX.benchmark import MethodConfig

# Filter by name
if scenario:
    scenarios = [s for s in scenarios if s.name == scenario]
    if not scenarios:
        print(f"No scenario matching '{scenario}'")
        sys.exit(1)

        
methods = [
    MethodConfig(
        name="grad_vanrossum",
        method_type="gradient",
        loss_type="van_rossum",
        loss_config=VanRossumLossConfig(tau_ms=15.0, weight_subthreshold=0.5, subthreshold_clamp_mv=-40),
        training_config=TrainingConfig(
            optimizer="polyak",
            learning_rate=0.01,
            n_epochs=250,
            print_every=10,
            surrogate_type="sigmoid",
            surrogate_slope=5.0,
            use_param_transform=True,
            clip_to_bounds=False,
            return_best=True,
            verbose=False,))
]

results = run_benchmark(
    scenarios=scenarios,
    methods=methods,
    output_dir=DEFAULT_RESULTS_DIR,
    resume=resume,
)

print(f"\nResults saved to: {DEFAULT_RESULTS_DIR}")


/Users/paulmayer/Projects/university/40_thesis/3_jaxley/jaxley/channels/non_capacitive/adex.py:82: UserWarning: The AdEx channel does not support surrogate gradients. Its gradient will be zero after every spike. Use AdExSurrogate for differentiable spiking.
  warn(



Results saved to: /Users/paulmayer/Projects/university/40_thesis/5_code/benchmarks/results_interactive


In [8]:

# Print summary
from ADoptEX.benchmark.analysis import print_table, summary_table

table = summary_table(results)
print_table(table)


scenario             | method         | n_runs | mean_gamma | best_gamma | std_gamma | mean_loss | best_loss | mean_wall
------------------------------------------------------------------------------------------------------------------------
tonic_15pct_membrane | grad_vanrossum | 10     | 0.0161     | 0.1278     | 0.0385    | 53.1839   | 10.9146   | 0.0810   
